In [1]:

### ===========START GIS TD - Update most recent inspection Date 210219.py ===============
#### ==============================================================================

# coding: utf-8

# ### For more information regarding updating feature services, see the following tutorial
# https://developers.arcgis.com/python/sample-notebooks/updating-features-in-a-feature-layer/


# Connect to the GIS
from arcgis.gis import GIS
from arcgis import features
import pandas as pd
from arcgis import geometry #use geometry module to project Long,Lat to X and Y
from copy import deepcopy
import re
import numpy as np
import datetime

In [2]:
print("Logging In...")
# Login using my login credentials
gis = GIS("https://www.arcgis.com","cara_moore","Tah0eesri")

Logging In...


In [3]:
# # Get GIS Feature service


print("Getting the feature service by ID...")

# Uncomment for actual TD Dspace, not test #!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
dspace_item = gis.content.get('ce554f7e3ac34d0cab808c72527e466d') # uncomment this for not a test, real deal.
# dspace_item = gis.content.get('b683c402ca6d4ce9b8347c8d8e000a5d') # Uncomment this for a test version

# ### Get feature layer

# In[27]:


print("Getting the properties layer from the Feature service...")
dspace_feature_layer = dspace_item.layers[0]


# ### Query feature layer


print("Querying the dspace feature set...")
dspace_feature_set = dspace_feature_layer.query() #querying without any conditions returns all the features

Getting the feature service by ID...
Getting the properties layer from the Feature service...
Querying the dspace feature set...


In [4]:
# ### Inspecting existing fields of the feature layer
# The `manager` property of the `FeatureLayer` object exposes a set of methods to read and update the properties and definition of feature layers.



#Get the existing list of fields on the cities feature layer
# dspace_property_fields = dspace_feature_layer.manager.properties.fields


# ## Get the dspace table from the feature service

print("Getting the space table from the feature service...")
dspace_table = dspace_item.tables[0]


# ### Setup inspection data into a table


print("Querying the inspection status...")
dspace_table_qry = dspace_table.query().sdf #querying without any conditions returns all the features
dspace_table_qry


# ### Pull out the most recent inspection status


dspace_table_qry.to_csv('dspace_table_qry_UpdateDate.csv')


print("Performing group by ParentGUID select newest inspection date...")
dspace_table_qry = dspace_table_qry.dropna(subset=['PARENTGUID'])  ### added this line to fix the "List likes to .loc error - use reindex instead"  # NOT SURE THIS FIXED IT
print("Expect warning here for list likes if it's not working.")
inspection_status = dspace_table_qry.loc[dspace_table_qry.groupby('PARENTGUID').INSPECTIONDATE.idxmax().dropna(),:] # I THINK ADDING .dropna() here fixed list likes warning.
print(inspection_status)

inspection_status.to_csv("C:\\DSI\inspection_status_UpdateDate.csv")



Getting the space table from the feature service...
Querying the inspection status...
Performing group by ParentGUID select newest inspection date...
Expect warning here for list likes if it's not working.
        OBJECTID Funding EligibleAcres InspectionCompliance  InspecNumeral  \
2378.0      2418    None          None            Compliant            1.0   
1953.0      1993    None          None            Compliant            1.0   
1828.0      1868    None          None            Compliant            1.0   
1933.0      1973    None          None            Compliant            1.0   
1485.0      1516    None          None            Compliant            1.0   
...          ...     ...           ...                  ...            ...   
1806.0      1846    None          None            Compliant            1.0   
2232.0      2272    None          None            Compliant            1.0   
1162.0      1192    None          None            Compliant            1.0   
220.0        2

In [5]:




# ### Setup property data into a dataframe


print('Reading in Properties data frame...')
df_properties = dspace_feature_set.sdf


# #### Identifying Overlapping Rows


df_properties.to_csv('df_properties_UpdateDate.csv')



df_properties['PARENTGUID'] = df_properties['GLOBALID']


overlap_rows = pd.merge(left = df_properties, right = inspection_status, how='inner',
                       on = ['PARENTGUID'])


# ### Filter out a dataframe that just contains the inspection DATES that need to be updated

print(overlap_rows)
overlap_rows.to_csv('C:\\DSI\overlaprows_UpdateDate.csv')

print("Overlap Rows completed")




Reading in Properties data frame...
      OBJECTID_x EligibleAcres_x InspectionCompliance_x  \
0             38            None              Compliant   
1             39            None              Compliant   
2             40            None              Compliant   
3             41            None              Compliant   
4             42            None              Compliant   
...          ...             ...                    ...   
1867        2064            None              Compliant   
1868        2065            None              Compliant   
1869        2066            None              Compliant   
1870        2069            None              Compliant   
1871        2070            None              Compliant   

     CALCULATED_INSPECTIONSTATUS INSPECTIONSTATUS_x            ADDRESSVISIBLE  \
0                           None               None  Yes - Without Reflective   
1                           None               None  Yes - Without Reflective   
2           

In [6]:

df_ = overlap_rows[(overlap_rows['INSPECTIONDATE'] != overlap_rows['CALCULATED_EDITED_DATE_x'])]
df_.to_csv('C:\\DSI\df_UpdateDate.csv')



In [7]:

# df_ = overlap_rows[(overlap_rows['InspectionCompliance_y'] != overlap_rows['InspectionCompliance_x'])]

# ### Create the list of features to be updated

# In[41]:


all_features = dspace_feature_set.features

features_for_update = [] #list that will contain corrected features

update_data = df_[['PARENTGUID','INSPECTIONDATE']].dropna()



# Create List for Update

In [8]:
for PARENTGUID in update_data['PARENTGUID']:
    # get the matching row from csv
    matching_row = update_data.where(update_data.PARENTGUID == PARENTGUID).dropna()
    
    print(str(PARENTGUID) + " Changing Date Status of: " + matching_row['PARENTGUID'].values[0] + " to " + str(matching_row['INSPECTIONDATE'].values[0])) 
    
     # get the feature to be updated
    original_feature = [f for f in all_features if f.attributes['GLOBALID'] == PARENTGUID][0]

    feature_to_be_updated = deepcopy(original_feature)
    feature_to_be_updated.attributes['CALCULATED_EDITED_DATE'] = pd.to_datetime(matching_row['INSPECTIONDATE'].values[0])
    print(feature_to_be_updated.attributes)
    print('Matching Row:')
    print(matching_row['INSPECTIONDATE'])
      #add this to the list of features to be updated
    features_for_update.append(feature_to_be_updated)
    
    
print("List of properties to update generated.  Update next.")
print(features_for_update)

List of properties to update generated.  Update next.
[]


In [9]:
# ### Update the feature


features_for_update


dspace_feature_layer.edit_features(updates= features_for_update)
# print("Inspection Status Updated.")


print("Meow")

Parameters not valid for edit_features
Meow
